# Capacity with Population Density

Estimate reported bed counts from Virtue Foundation facility features and population density.
This version uses the facilities table and population density.

Open this notebook in the project Git folder. Enter a density table or file in
the configuration cell. The supplied facility labels and the seed define the
same facility-group split in both variants. Compare the split fingerprints
before comparing metrics. See the [data handoff](CLAUDE_HANDOFF.md) for input formats.

In [ ]:
%pip install "scikit-learn==1.7.2" "wandb==0.31.2" "skops==0.16.0" "safetensors==0.8.0"

In [ ]:
import os
import sys
from pathlib import Path

INCLUDE_NFHS = False
SEED, EPOCHS, PATIENCE = 42, 300, 30
CATALOG_SCHEMA = "databricks_virtue_foundation_dataset_dais_2026.virtue_foundation_dataset"
FACILITIES_TABLE = f"{CATALOG_SCHEMA}.facilities"
POSTAL_TABLE = f"{CATALOG_SCHEMA}.india_post_pincode_directory"
NFHS_TABLE = f"{CATALOG_SCHEMA}.nfhs_5_district_health_indicators"

# Set exactly one input. A path can be a Spark-readable Unity Catalog volume.
DENSITY_TABLE = ""
DENSITY_PATH = ""
DENSITY_FORMAT = "csv"  # csv or parquet
DENSITY_KEY = "district"  # district or pin
DENSITY_COLUMNS = {
    "state_column": "state",
    "district_column": "district",
    "density_column": "population_density",
}
DENSITY_SOURCE = ""
DENSITY_VINTAGE = ""

cwd = Path.cwd()
relative = Path("ML/models with population density")
repo_override = os.environ.get("DATATHON_REPO_ROOT")
candidates = [Path(repo_override).expanduser()] if repo_override else [cwd, *cwd.parents]
root = next((path for path in candidates if (path / relative / "density_data.py").is_file()), None)
if root is None:
    raise RuntimeError("Open the notebook in the project Git folder or set DATATHON_REPO_ROOT.")
sys.path[:0] = [str(root / relative), str(root / "ML/notebooks")]

from density_capacity import TEXT, train_capacity
from density_data import join_nfhs, join_population_density, prepare_district_keys
from wandb_models import log_torch_capacity

## Match population data to the facility

Use a district and state or a PIN. Missing and ambiguous joins stay visible.
Neither the density join nor the NFHS join drops unmatched facilities.
The model imputes missing numerical inputs from the training facilities.

In [ ]:
if bool(DENSITY_TABLE) == bool(DENSITY_PATH):
    raise ValueError("Set exactly one of DENSITY_TABLE and DENSITY_PATH.")
if DENSITY_KEY not in ("district", "pin"):
    raise ValueError("DENSITY_KEY must be district or pin.")
if DENSITY_TABLE:
    density = spark.table(DENSITY_TABLE).toPandas()
elif DENSITY_FORMAT == "csv":
    density = spark.read.option("header", "true").csv(DENSITY_PATH).toPandas()
elif DENSITY_FORMAT == "parquet":
    density = spark.read.parquet(DENSITY_PATH).toPandas()
else:
    raise ValueError("DENSITY_FORMAT must be csv or parquet.")

facilities = spark.table(FACILITIES_TABLE).toPandas()
district_keys = None
if INCLUDE_NFHS or DENSITY_KEY == "district":
    postal = spark.table(POSTAL_TABLE).select("pincode", "district", "statename").toPandas()
    district_keys = prepare_district_keys(facilities, postal)
training, density_report = join_population_density(
    facilities, density, keys=district_keys, key=DENSITY_KEY, **DENSITY_COLUMNS,
)
display(density_report)
if INCLUDE_NFHS:
    nfhs = spark.table(NFHS_TABLE).toPandas()
    training, nfhs_report = join_nfhs(training, nfhs, keys=district_keys)
    display(nfhs_report)

## Fit and compare on held-out facilities

Fit the preprocessor and MLP on the training partition. Validation selects the
training checkpoint. Report MAE, RMSE and R² in beds for the held-out partition,
and check error with and without a usable density match. The training median gives
a simple comparison on the same rows.

In [ ]:
result = train_capacity(training, include_nfhs=INCLUDE_NFHS, seed=SEED, epochs=EPOCHS, patience=PATIENCE)
print("Facility groups:", result["facility_counts"])
print("Split fingerprint:", result["split_fingerprint"])
print("Training median baseline:", result["baseline_metrics"])
display(result["test_cohorts"])

## Save the evaluated model in Weights & Biases

The artifact contains the fitted preprocessor, network weights and aggregate
evaluation metrics. The default W&B mode is offline. See the [tracking setup](../README.md).

In [ ]:
source_tables = [FACILITIES_TABLE]
if district_keys is not None:
    source_tables.append(POSTAL_TABLE)
if INCLUDE_NFHS:
    source_tables.append(NFHS_TABLE)
capacity_run = log_torch_capacity(
    name="capacity-mlp-density-nfhs" if INCLUDE_NFHS else "capacity-mlp-density",
    model=result["model"],
    preprocessor=result["preprocessor"],
    numeric_features=result["numeric_features"],
    categorical_features=result["categorical_features"],
    metrics=result["test_metrics"],
    baseline_metrics=result["baseline_metrics"],
    config={
        "source_tables": source_tables,
        "text_features": list(TEXT),
        "include_nfhs": INCLUDE_NFHS,
        "density_source": DENSITY_SOURCE or DENSITY_TABLE or "density file",
        "density_vintage": DENSITY_VINTAGE,
        "density_join_key": DENSITY_KEY,
        "density_columns": DENSITY_COLUMNS,
        "split_seed": SEED,
        "split_fingerprint": result["split_fingerprint"],
        "facility_counts": result["facility_counts"],
        "row_counts": result["row_counts"],
        "test_cohorts": result["test_cohorts"].where(result["test_cohorts"].notna(), None)
                                            .to_dict(orient="records"),
        "prediction_postprocess": "expm1, then round to nearest whole bed and clamp at zero",
    },
)
print(capacity_run)